# Movie Recommender System

Content-based recommender: every movie is turned into a bag of *tags* (overview words, genres,
keywords, top 3 cast members and the director), vectorised with `CountVectorizer`, and compared
with cosine similarity. `recommend(title)` returns the most similar movies.

**Data:** [TMDB 5000 Movie Dataset](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata).
`tmdb_5000_movies.csv` is in the repo; download `tmdb_5000_credits.csv` from Kaggle and put it next
to this notebook to include cast and director (the notebook still runs without it).

In [1]:
import ast
import os

import numpy as np
import pandas as pd

In [2]:
movies = pd.read_csv('tmdb_5000_movies.csv')
HAS_CREDITS = os.path.exists('tmdb_5000_credits.csv')
if HAS_CREDITS:
    credits = pd.read_csv('tmdb_5000_credits.csv')
else:
    print('tmdb_5000_credits.csv not found - continuing without cast and director')

tmdb_5000_credits.csv not found - continuing without cast and director


In [3]:
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [4]:
movies.shape

(4803, 20)

### Merge movies with credits

Merge on the TMDB **id**, not on `title`: some titles are shared by different films
(*Batman*, *The Host*, *Out of the Blue*), and a title merge pairs each copy with every other copy,
mixing up one film's overview with another film's cast.

In [5]:
movies[movies['title'].duplicated(keep=False)][['id', 'title', 'release_date']]

,id,title,release_date
972,72710,The Host,2013-03-22
1359,268,Batman,1989-06-23
2877,1255,The Host,2006-07-27
3647,39269,Out of the Blue,1980-05-01
3693,10844,Out of the Blue,2006-10-12
4267,2661,Batman,1966-07-30


In [6]:
if HAS_CREDITS:
    movies = movies.merge(credits.drop(columns=['title']), left_on='id', right_on='movie_id')
else:
    movies['cast'] = '[]'
    movies['crew'] = '[]'
movies.shape

(4803, 22)

In [7]:
# Keep only the columns used to build tags
movies = movies[['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']].copy()
movies.head()

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",[],[]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",[],[]
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",[],[]
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",[],[]
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",[],[]


In [8]:
movies.isnull().sum()

id          0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [9]:
movies.dropna(inplace=True)
# dropna leaves gaps in the index; reset it so a row's label equals its position
# in the similarity matrix built below.
movies.reset_index(drop=True, inplace=True)
movies.shape

(4800, 7)

### Turn the JSON columns into lists of names

In [10]:
def convert(text):
    return [i['name'] for i in ast.literal_eval(text)]


def convert3(text):
    # top 3 cast members
    return convert(text)[:3]


def fetch_director(text):
    return [i['name'] for i in ast.literal_eval(text) if i['job'] == 'Director']

In [11]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert3)
movies['crew'] = movies['crew'].apply(fetch_director)
movies.head()

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...",[],[]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...",[],[]
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...",[],[]
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...",[],[]
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...",[],[]


In [12]:
# 'Science Fiction' -> 'ScienceFiction' so multi-word names stay a single tag
def collapse(L):
    return [i.replace(" ", "") for i in L]


for col in ['genres', 'keywords', 'cast', 'crew']:
    movies[col] = movies[col].apply(collapse)
movies.sample(5, random_state=0)

,id,title,overview,genres,keywords,cast,crew
2255,19457,The Yards,"In the rail yards of Queens, contractors repai...","[Drama, Action, Thriller, Crime]","[corruption, subway, infiltration, criminal]",[],[]
608,3093,Basic Instinct 2,Novelist Catherine Tramell is once again in tr...,"[Crime, Mystery, Thriller]","[malenudity, sex, legs, soholondon, playinggod...",[],[]
2856,14976,Rachel Getting Married,A young woman who has been in and out from reh...,[Drama],"[sister, independentfilm, maidofhonor]",[],[]
1889,9469,He Got Game,A basketball player's father must try to convi...,[Drama],"[prison, fathersonrelationship, homicide, begn...",[],[]
1519,3587,Bogus,"Recently orphaned, a young boy is taken in by ...","[Fantasy, Comedy, Family]","[circus, magic, aunt, imaginaryfriend]",[],[]


In [13]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
new = movies[['id', 'title', 'tags']].copy()
new['tags'] = new['tags'].apply(lambda x: " ".join(x).lower())
new.head()

,id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


### Stemming

Reduces words to their root so *action*, *actions* and *acting* count as the same tag.

In [14]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()


def stem(text):
    return " ".join(ps.stem(word) for word in text.split())


new['tags'] = new['tags'].apply(stem)
new['tags'][0]

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d'

### Vectorise and compute similarity

In [15]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words='english')
vector = cv.fit_transform(new['tags'])  # sparse; no need for .toarray()
vector.shape

(4800, 5000)

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vector)
similarity.shape

(4800, 4800)

In [17]:
def recommend(movie, n=5):
    matches = new.index[new['title'].str.lower() == movie.lower()]
    if len(matches) == 0:
        raise ValueError(f"Movie not found: {movie!r}")
    index = matches[0]
    distances = sorted(enumerate(similarity[index]), reverse=True, key=lambda x: x[1])
    return [new.iloc[i].title for i, _ in distances if i != index][:n]

In [18]:
recommend('Frozen')

['Spirit: Stallion of the Cimarron',
 'Aladdin',
 'Delgo',
 'Jungle Shuffle',
 'Pocahontas']

In [19]:
recommend('The Dark Knight')

['The Dark Knight Rises',
 'Batman Begins',
 'Batman Returns',
 'Batman Forever',
 'Batman']

In [20]:
# Movies near the end of the table used to return the wrong row or raise IndexError
recommend(new['title'].iloc[-1])

['Bad Grandpa',
 'How to Fall in Love',
 'The R.M.',
 'After Earth',
 'The Straight Story']

### Export for the web app

`model/build_recommendations.py` runs this same pipeline and writes `web/data/movies.json`,
which the frontend in `web/` loads. Re-run it after changing the pipeline:

```bash
python model/build_recommendations.py
```